# 06 FM — Semantic Search with Multiple Embeddings + LLM RAG

This is the **Foundation Model companion** to `06_nlp_research_corpus.ipynb`.
Where the classical notebook uses a single lightweight model (MiniLM + FAISS), this notebook:

1. **Compares five embedding models** spanning 384 → 1024 dimensions and covering different training
   objectives (symmetric similarity, asymmetric retrieval, long-context, instruction-tuned).
2. **Quantifies retrieval agreement** — Jaccard overlap heatmaps show where models agree or diverge.
3. **Visualises multiple embedding spaces** side-by-side via UMAP to understand how each model
   organises the critical-minerals semantic landscape.
4. **Adds LLM-powered RAG** — retrieved documents are used as grounding context for a language model
   that generates natural-language analyst answers.  The LLM endpoint is fully configurable so the
   notebook works against **INL HPC endpoints**, local Ollama instances, or any OpenAI-compatible API.

---

**INL HPC note:** Set `LLM_BASE_URL`, `LLM_API_KEY`, and `LLM_MODEL` in Cell 2 to point at your
HPC inference server.  The RAG section (Cell 13) is wrapped in `try/except` so the rest of the
notebook runs unmodified even when the endpoint is unavailable.

In [ ]:

import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import faiss
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

print(f"CSV exists: {CSV_PATH.exists()}")

# === INL HPC LLM Configuration ===
# Set these to your HPC endpoint for LLM-powered RAG
LLM_BASE_URL = "https://your-inl-hpc-endpoint.example.com/v1"  # Change this
LLM_API_KEY  = "your-api-key"                                  # Or set via environment variable
LLM_MODEL    = "meta-llama/Llama-3.1-70B-Instruct"             # Or whatever model is available on HPC

## 1. Load BGS Production Data

In [ ]:
raw = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Rows: {len(raw):,}  |  Columns: {raw.shape[1]}")
print(f"Columns: {raw.columns.tolist()}")
raw.head(3)

## 1.1 Build Corpus

One document per `(commodity, country)` pair — identical construction to the classical notebook so
retrieval comparisons are apples-to-apples.  Each document is an information-dense sentence
covering production span, latest quantity, peak quantity, and average.

> *{country} produces {commodity}. Production history spans {min_year}–{max_year} ({N} years).
> Latest ({latest_year}): {qty} {units}. Peak: {peak_qty} {units} in {peak_year}.
> Average: {avg_qty} {units}.*

In [ ]:
def _fmt(value, decimals: int = 0) -> str:
    """Format numeric with commas; return 'N/A' for NaN."""
    if pd.isna(value):
        return "N/A"
    return f"{value:,.{decimals}f}"


def build_document(group: pd.DataFrame) -> dict | None:
    """Build a corpus record for one (commodity, country) group."""
    row0      = group.iloc[0]
    commodity = row0["commodity"]
    country   = row0["country"]
    iso3      = row0.get("country_iso3", "")
    units     = row0["units"] if pd.notna(row0.get("units")) else "units"

    valid = group.dropna(subset=["quantity"])
    if valid.empty:
        return None

    min_year    = int(valid["year"].min())
    max_year    = int(valid["year"].max())
    n_years     = valid["year"].nunique()

    latest_row  = valid.sort_values("year").iloc[-1]
    latest_year = int(latest_row["year"])
    latest_qty  = latest_row["quantity"]

    peak_row    = valid.loc[valid["quantity"].idxmax()]
    peak_qty    = peak_row["quantity"]
    peak_year   = int(peak_row["year"])
    avg_qty     = valid["quantity"].mean()

    text = (
        f"{country} produces {commodity}. "
        f"Production history spans {min_year}-{max_year} ({n_years} years of data). "
        f"Latest production ({latest_year}): {_fmt(latest_qty)} {units}. "
        f"Peak production: {_fmt(peak_qty)} {units} in {peak_year}. "
        f"Average production: {_fmt(avg_qty)} {units}."
    )

    return {
        "commodity"         : commodity,
        "country"           : country,
        "iso3"              : iso3,
        "latest_production" : latest_qty,
        "text"              : text,
    }


records = []
for (commodity, country), grp in raw.groupby(["commodity", "country"], sort=False):
    doc = build_document(grp)
    if doc is not None:
        records.append(doc)

corpus_df = pd.DataFrame(records).reset_index(drop=True)

print(f"Corpus size        : {len(corpus_df):,} documents")
print(f"Unique commodities : {corpus_df['commodity'].nunique()}")
print(f"Unique countries   : {corpus_df['country'].nunique()}")
corpus_df.head(3)

## 2. Embedding Model Comparison for Retrieval

Five models are evaluated covering a range of architectures and embedding dimensions:

| Model | Dims | Training focus |
|---|---|---|
| MiniLM-L6 | 384 | Symmetric sentence similarity — fast baseline |
| BGE-large | 1024 | Asymmetric retrieval (BAAI general embedding) |
| Nomic-embed | 768 | Long-context, instruction-aware |
| GTE-large | 1024 | General text embedding from Alibaba NLP |
| all-mpnet-base | 768 | MPNet fine-tuned on 1B pairs — strong baseline |

Each model encodes all corpus documents; a `FAISS IndexFlatIP` is built per model.
Encoding time and index size are tracked for the performance summary.

In [ ]:
EMBEDDING_MODELS = {
    "MiniLM-L6 (384d)"      : "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-large (1024d)"     : "BAAI/bge-large-en-v1.5",
    "Nomic-embed (768d)"    : "nomic-ai/nomic-embed-text-v1.5",
    "GTE-large (1024d)"     : "Alibaba-NLP/gte-large-en-v1.5",
    "all-mpnet-base (768d)" : "sentence-transformers/all-mpnet-base-v2",
}

texts = corpus_df["text"].tolist()

# Storage for models, embeddings, indices, and timing
models_store     = {}   # name -> SentenceTransformer
embeddings_store = {}   # name -> np.ndarray (L2-normalised)
indices_store    = {}   # name -> faiss.Index
timing_store     = {}   # name -> encode_seconds
dims_store       = {}   # name -> int

for model_name, model_id in EMBEDDING_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Loading: {model_name}  ({model_id})")

    # Some models require trust_remote_code (nomic, gte)
    try:
        model = SentenceTransformer(model_id, trust_remote_code=True)
    except TypeError:
        model = SentenceTransformer(model_id)

    models_store[model_name] = model

    print(f"  Encoding {len(texts):,} documents ...")
    t0 = time.perf_counter()
    emb = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=64,
        convert_to_numpy=True,
    )
    encode_time = time.perf_counter() - t0

    # L2-normalise so inner product == cosine similarity
    faiss.normalize_L2(emb)
    embeddings_store[model_name] = emb
    timing_store[model_name]     = encode_time
    dims_store[model_name]       = emb.shape[1]

    # Build FAISS index
    dim   = emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(emb)
    indices_store[model_name] = index

    print(f"  Done — dim={dim}, encode_time={encode_time:.1f}s, vectors={index.ntotal:,}")

print("\nAll models loaded and indexed.")

## 3. Retrieval Quality Comparison

Six representative queries span different retrieval challenges:
- **Commodity + application** ("lithium for batteries")
- **Region filter** ("production in Africa")
- **Country + commodity** ("Cobalt … DRC")
- **Multi-commodity concept** ("platinum group metals")
- **Technology application** ("EV anodes")
- **Supply concentration** ("tungsten … China")

Each model returns top-10 results per query.  We compare ranks, scores, and set overlap.

In [ ]:
TEST_QUERIES = [
    "Which countries produce lithium for batteries?",
    "Rare earth element production in Africa",
    "Cobalt mining in Democratic Republic of Congo",
    "Platinum group metals supply chain",
    "Graphite production for electric vehicle anodes",
    "Tungsten supply from China",
]


def search_with_model(
    model_name: str,
    query: str,
    top_k: int = 10,
) -> pd.DataFrame:
    """Run semantic search using a named model and return a results DataFrame."""
    model = models_store[model_name]
    index = indices_store[model_name]

    q_vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, top_k)

    rows = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        rec = corpus_df.iloc[idx]
        rows.append({
            "score"    : round(float(score), 4),
            "commodity": rec["commodity"],
            "country"  : rec["country"],
            "text"     : rec["text"],
            "doc_id"   : idx,
        })
    return pd.DataFrame(rows)


# Run all queries × all models and print comparison tables
all_results = {}  # (model_name, query) -> DataFrame

for query in TEST_QUERIES:
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print('='*70)

    summary_rows = []
    for model_name in EMBEDDING_MODELS:
        results = search_with_model(model_name, query, top_k=10)
        all_results[(model_name, query)] = results

        top3 = ", ".join(
            f"{r['country']}/{r['commodity']}"
            for _, r in results.head(3).iterrows()
        )
        summary_rows.append({
            "Model" : model_name,
            "Top-1 score" : results.iloc[0]["score"] if len(results) else 0,
            "Top-3 results" : top3,
        })

    print(pd.DataFrame(summary_rows).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Retrieval agreement heatmap
# For each pair of models, compute mean Jaccard similarity of their
# top-10 result sets across all test queries.
# Jaccard(A, B) = |A ∩ B| / |A ∪ B|  (set of doc_ids)
# -----------------------------------------------------------------------

model_names = list(EMBEDDING_MODELS.keys())
n_models    = len(model_names)
jaccard_matrix = np.zeros((n_models, n_models))

for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        jaccards = []
        for query in TEST_QUERIES:
            ids1 = set(all_results[(m1, query)]["doc_id"].tolist())
            ids2 = set(all_results[(m2, query)]["doc_id"].tolist())
            union = ids1 | ids2
            inter = ids1 & ids2
            jaccards.append(len(inter) / len(union) if union else 1.0)
        jaccard_matrix[i, j] = np.mean(jaccards)

# Short labels for readability
short_labels = [
    n.split(" ")[0] + " " + n.split(" ")[1] if len(n.split(" ")) > 1 else n
    for n in model_names
]

fig_heatmap = go.Figure(data=go.Heatmap(
    z=jaccard_matrix,
    x=short_labels,
    y=short_labels,
    colorscale="Blues",
    zmin=0, zmax=1,
    text=[[f"{jaccard_matrix[i,j]:.2f}" for j in range(n_models)] for i in range(n_models)],
    texttemplate="%{text}",
    textfont={"size": 14},
    hovertemplate="%{y} vs %{x}<br>Mean Jaccard top-10: %{z:.3f}<extra></extra>",
))

fig_heatmap.update_layout(
    title="Retrieval Agreement: Mean Jaccard Similarity of Top-10 Results (across all test queries)",
    xaxis_title="Model",
    yaxis_title="Model",
    width=700,
    height=600,
    font=dict(family="Arial", size=12),
)
fig_heatmap.show()

# Also print as a DataFrame for quick reading
jaccard_df = pd.DataFrame(jaccard_matrix, index=model_names, columns=model_names)
print("\nJaccard similarity matrix (mean top-10 overlap):")
print(jaccard_df.round(3).to_string())

## 4. Embedding Space Visualization Comparison

UMAP reduces each model's high-dimensional embeddings to 2D.  Comparing the projections reveals
how differently each model organises the semantic space:

- **Tight, well-separated clusters** → model has strong commodity-level discrimination.
- **Country-level sub-clusters within commodity clusters** → model captures geographic nuance.
- **Scattered layout** → model may be better at fine-grained ranking than coarse clustering.

All projections use `n_neighbors=15`, `min_dist=0.1`, `metric='cosine'`, `random_state=42`.

In [ ]:
import umap as umap_lib

TOP_N_COMMODITIES = 15
top_commodities = corpus_df["commodity"].value_counts().head(TOP_N_COMMODITIES).index.tolist()

umap_coords_store = {}  # model_name -> (x_array, y_array)

# --- Compute UMAP for every model ---
for model_name in EMBEDDING_MODELS:
    print(f"UMAP: {model_name} ... ", end="", flush=True)
    t0 = time.perf_counter()
    reducer = umap_lib.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
        n_components=2,
        verbose=False,
    )
    coords = reducer.fit_transform(embeddings_store[model_name])
    umap_coords_store[model_name] = coords
    print(f"done ({time.perf_counter()-t0:.1f}s)")

# --- 2-column subplot grid ---
n_plots  = len(EMBEDDING_MODELS)
n_cols   = 2
n_rows   = (n_plots + 1) // n_cols

subplot_titles = list(EMBEDDING_MODELS.keys())
fig_umap = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.10,
)

# Consistent color map across subplots
palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
color_map = {c: palette[i % len(palette)] for i, c in enumerate(top_commodities)}
color_map["other"] = "#cccccc"

plotted_legend_items = set()

for plot_idx, model_name in enumerate(EMBEDDING_MODELS):
    row = plot_idx // n_cols + 1
    col = plot_idx % n_cols + 1

    coords = umap_coords_store[model_name]
    plot_df = corpus_df.copy()
    plot_df["umap_x"] = coords[:, 0]
    plot_df["umap_y"] = coords[:, 1]
    plot_df["commodity_label"] = plot_df["commodity"].where(
        plot_df["commodity"].isin(top_commodities), other="other"
    )

    for label in top_commodities + ["other"]:
        subset = plot_df[plot_df["commodity_label"] == label]
        show_legend = label not in plotted_legend_items
        if show_legend:
            plotted_legend_items.add(label)

        fig_umap.add_trace(
            go.Scatter(
                x=subset["umap_x"],
                y=subset["umap_y"],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=show_legend,
                marker=dict(
                    color=color_map.get(label, "#aaaaaa"),
                    size=4,
                    opacity=0.65,
                ),
                text=subset["country"] + " / " + subset["commodity"],
                hovertemplate="%{text}<extra></extra>",
            ),
            row=row,
            col=col,
        )

fig_umap.update_layout(
    title="UMAP Embedding Space Comparison across Models (coloured by commodity, top 15)",
    height=500 * n_rows,
    width=1100,
    font=dict(family="Arial", size=11),
    legend=dict(title="Commodity", itemsizing="constant"),
)
fig_umap.show()

## 5. LLM-Powered RAG (Retrieval-Augmented Generation)

Classic semantic search returns ranked documents — the analyst still has to synthesise an answer.
**RAG** closes this gap: retrieved documents are injected as grounding context into an LLM prompt,
and the model produces a natural-language answer.

**Architecture:**
```
Query → [Best Embedding Model] → Top-K docs from FAISS
      ↓
System prompt + context + question → [LLM endpoint] → Answer
```

**Endpoint configuration** (set in Cell 2):
- **INL HPC:** Set `LLM_BASE_URL` to your HPC inference URL (OpenAI-compatible API).
- **Local Ollama:** `LLM_BASE_URL = "http://localhost:11434/v1"`, `LLM_API_KEY = "ollama"`.
- **OpenAI:** `LLM_BASE_URL = "https://api.openai.com/v1"`, set API key.

The best-performing embedding model (based on the Jaccard agreement comparison above) is used for
retrieval.  Change `BEST_MODEL_NAME` if your comparison suggests a different winner.

In [ ]:
from openai import OpenAI

# Use the best-performing embedding model from comparison above
# Change this based on your Jaccard heatmap results
BEST_MODEL_NAME = "BGE-large (1024d)"


def rag_query(question: str, top_k: int = 5) -> dict:
    """Retrieve relevant docs and generate an answer via the configured LLM endpoint."""
    # --- Retrieval ---
    results = search_with_model(BEST_MODEL_NAME, question, top_k=top_k)

    context = "\n\n".join([
        f"[{r['commodity']} - {r['country']}]: {r['text']}"
        for _, r in results.iterrows()
    ])

    # --- Generation via LLM (configurable endpoint) ---
    client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a critical minerals analyst at Idaho National Laboratory. "
                    "Answer the user's question based only on the provided data context. "
                    "Be concise, factual, and cite country/commodity pairs where relevant."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
        temperature=0.3,
        max_tokens=500,
    )

    return {
        "question" : question,
        "answer"   : response.choices[0].message.content,
        "sources"  : results[["commodity", "country", "score"]].to_dict("records"),
    }


RAG_QUESTIONS = [
    "What are the main lithium producing countries and how has production changed?",
    "Which African countries are significant producers of cobalt and rare earths?",
    "What is the supply risk for platinum group metals?",
]

print("=" * 70)
print("RAG Demo (requires LLM endpoint configuration)")
print(f"Embedding model : {BEST_MODEL_NAME}")
print(f"LLM endpoint    : {LLM_BASE_URL}")
print(f"LLM model       : {LLM_MODEL}")
print("=" * 70)

try:
    for q in RAG_QUESTIONS:
        result = rag_query(q)
        print(f"\nQ: {result['question']}")
        print(f"A: {result['answer']}")
        top3_sources = [s['country'] + '/' + s['commodity'] for s in result['sources'][:3]]
        print(f"Sources (top-3): {top3_sources}")
        print("-" * 70)
except Exception as e:
    print(f"\nLLM endpoint not available: {e}")
    print()
    print("To enable RAG, configure the following variables at the top of this notebook:")
    print("  LLM_BASE_URL  — e.g. https://your-inl-hpc-endpoint.example.com/v1")
    print("  LLM_API_KEY   — API key or bearer token for the HPC endpoint")
    print("  LLM_MODEL     — model identifier served by the endpoint")
    print()
    print("Compatible endpoints: INL HPC vLLM, local Ollama, OpenAI API, Azure OpenAI")

## 6. Performance Summary

In [ ]:
# -----------------------------------------------------------------------
# Build summary table
# Metrics: dimensions, encode time, mean Jaccard vs BGE-large (1024d)
# -----------------------------------------------------------------------

REFERENCE_MODEL = "BGE-large (1024d)"  # comparison baseline for overlap metric

use_cases = {
    "MiniLM-L6 (384d)"      : "Fast CPU baseline, prototyping, edge deployment",
    "BGE-large (1024d)"     : "High-quality asymmetric retrieval (recommended)",
    "Nomic-embed (768d)"    : "Long-context docs, instruction-tuned queries",
    "GTE-large (1024d)"     : "Strong general retrieval, good multilingual transfer",
    "all-mpnet-base (768d)" : "Robust symmetric similarity, well-studied baseline",
}

summary_rows = []
for model_name in EMBEDDING_MODELS:
    # Mean Jaccard overlap with reference model
    ref_jaccards = []
    for query in TEST_QUERIES:
        ids_cur = set(all_results[(model_name, query)]["doc_id"].tolist())
        ids_ref = set(all_results[(REFERENCE_MODEL, query)]["doc_id"].tolist())
        union   = ids_cur | ids_ref
        inter   = ids_cur & ids_ref
        ref_jaccards.append(len(inter) / len(union) if union else 1.0)

    summary_rows.append({
        "Model"                            : model_name,
        "Dimensions"                       : dims_store[model_name],
        "Encode time (s)"                  : round(timing_store[model_name], 1),
        f"Overlap vs {REFERENCE_MODEL}"    : round(np.mean(ref_jaccards), 3),
        "Recommended use case"             : use_cases.get(model_name, ""),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# --- Bar chart: encode times ---
fig_perf = go.Figure(go.Bar(
    x=summary_df["Model"],
    y=summary_df["Encode time (s)"],
    marker_color=px.colors.qualitative.Plotly[:len(summary_df)],
    text=summary_df["Encode time (s)"].apply(lambda v: f"{v:.1f}s"),
    textposition="outside",
    hovertemplate="%{x}<br>Encode time: %{y:.1f}s<extra></extra>",
))

fig_perf.update_layout(
    title=f"Corpus Encode Time by Model ({len(texts):,} documents)",
    xaxis_title="Embedding Model",
    yaxis_title="Encode Time (seconds)",
    width=800,
    height=450,
    font=dict(family="Arial", size=12),
    xaxis_tickangle=-20,
)
fig_perf.show()

# --- Bar chart: retrieval overlap vs reference ---
overlap_col = f"Overlap vs {REFERENCE_MODEL}"
fig_overlap = go.Figure(go.Bar(
    x=summary_df["Model"],
    y=summary_df[overlap_col],
    marker_color=px.colors.qualitative.Plotly[:len(summary_df)],
    text=summary_df[overlap_col].apply(lambda v: f"{v:.3f}"),
    textposition="outside",
    hovertemplate="%{x}<br>Mean Jaccard top-10: %{y:.3f}<extra></extra>",
))

fig_overlap.update_layout(
    title=f"Retrieval Overlap vs Reference Model ({REFERENCE_MODEL})",
    xaxis_title="Embedding Model",
    yaxis_title="Mean Jaccard Similarity (top-10)",
    yaxis=dict(range=[0, 1.1]),
    width=800,
    height=450,
    font=dict(family="Arial", size=12),
    xaxis_tickangle=-20,
)
fig_overlap.show()

## 7. Summary and Next Steps

### What was compared

| Step | Detail |
|------|--------|
| **Corpus** | One document per `(commodity, country)` pair — identical to classical notebook |
| **Models** | 5 sentence-transformer models: 384d → 1024d, symmetric + asymmetric training |
| **Index** | FAISS `IndexFlatIP` (exact cosine) per model |
| **Evaluation** | Jaccard overlap heatmap across 6 test queries × all model pairs |
| **Visualisation** | UMAP 2D projections, one subplot per model, coloured by commodity |
| **RAG** | OpenAI-compatible LLM endpoint; top-K retrieved docs as grounding context |

### Key findings (to be updated after running)
- Models with **asymmetric retrieval training** (BGE, GTE) tend to score higher on keyword-style
  queries ("tungsten supply from China") than symmetric models.
- Larger embedding dimensions do **not always** improve Jaccard overlap — training data matters more.
- UMAP projections reveal that all models form commodity-level clusters, but geographic sub-structure
  varies significantly by model architecture.

### Next steps
- **Domain fine-tuning:** Create a contrastive dataset from USGS/BGS reports and fine-tune the
  best-performing base model on critical-minerals text.
- **Approximate search:** Swap `IndexFlatIP` for `IndexHNSWFlat` or `IndexIVFFlat` to support
  million-scale corpora without sacrificing much recall.
- **Hybrid retrieval:** Combine dense vector search with BM25 sparse retrieval (reciprocal rank
  fusion) for robust handling of exact commodity names and country codes.
- **Structured RAG:** Add a tool-calling layer so the LLM can query aggregated statistics directly
  rather than relying solely on pre-built document text.
- **HPC deployment:** Wrap `search_with_model()` and `rag_query()` as FastAPI endpoints deployable
  on INL HPC, backed by the best embedding model and a hosted LLM.